In [ ]:
import os
import glob
import numpy as np
import scipy.io as sio
import cv2
from scipy.ndimage import uniform_filter, sobel

# ==========================================
# 1. Core Model Implementation
# ==========================================
class GaussianNaiveBayesScratch:
    def __init__(self, var_smoothing=1e-4):
        self.var_smoothing = var_smoothing
        self.classes, self.priors = None, None
        self.means, self.vars = None, None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        self.priors = np.zeros(n_classes)
        self.means = np.zeros((n_classes, n_features))
        self.vars = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.priors[idx] = X_c.shape[0] / float(n_samples)
            self.means[idx, :] = np.mean(X_c, axis=0)
            self.vars[idx, :] = np.var(X_c, axis=0) + self.var_smoothing
        return self

    def _calc_log_likelihood(self, class_idx, X):
        m, v = self.means[class_idx], self.vars[class_idx]
        lp = np.log(self.priors[class_idx])
        t1 = -0.5 * np.sum(np.log(2.0 * np.pi * v))
        t2 = -0.5 * np.sum(((X - m)**2) / v, axis=1)
        return lp + t1 + t2

    def predict_proba(self, X):
        ll = np.array([self._calc_log_likelihood(i, X)
                       for i in range(len(self.classes))]).T
        m = np.max(ll, axis=1, keepdims=True)
        lse = m + np.log(np.sum(np.exp(ll - m), axis=1, keepdims=True))
        return np.exp(ll - lse)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

# ==========================================
# 2. Feature Extraction
# ==========================================
def extract_pixel_features(gray):
    gx = sobel(gray, axis=1); gy = sobel(gray, axis=0)
    grad_mag = np.sqrt(gx**2 + gy**2)
    local_mean = uniform_filter(gray, size=3)
    local_sq = uniform_filter(gray**2, size=3)
    local_std = np.sqrt(np.clip(local_sq - local_mean**2, 0, None))
    return np.stack([gray, grad_mag, gx, gy, local_mean, local_std], axis=-1)

# ==========================================
# 3. Data Loading & Sampling
# ==========================================
def load_bsds500_data(image_dir, gt_dir, max_images=10):
    '''Loads images, extracts features, and creates ground truth boundary labels'''
    X_all, y_all = [], []
    if not os.path.exists(image_dir) or not os.path.exists(gt_dir):
        print(f"Warning: Dataset directories not found.")
        return np.array([]), np.array([])
        
    img_paths = glob.glob(os.path.join(image_dir, '*.jpg'))[:max_images]
    for img_path in img_paths:
        base = os.path.splitext(os.path.basename(img_path))[0]
        gt_path = os.path.join(gt_dir, base + '.mat')
        if not os.path.exists(gt_path): continue
            
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None: continue
            
        features = extract_pixel_features(img)
        mat = sio.loadmat(gt_path)
        annotators = mat['groundTruth'][0]
        boundaries = np.stack([anno[0][0][1] for anno in annotators], axis=0)
        consensus = (np.mean(boundaries, axis=0) >= 0.5).astype(int)
        
        X_all.append(features.reshape(-1, features.shape[-1]))
        y_all.append(consensus.reshape(-1))
        
    return np.vstack(X_all), np.concatenate(y_all)

def balanced_sampling(X, y, samples_per_class=250):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    if len(pos_idx) > samples_per_class: pos_idx = np.random.choice(pos_idx, samples_per_class, replace=False)
    if len(neg_idx) > samples_per_class: neg_idx = np.random.choice(neg_idx, samples_per_class, replace=False)
    idx = np.concatenate([pos_idx, neg_idx])
    np.random.shuffle(idx)
    return X[idx], y[idx]

# ==========================================
# 4. Evaluation Metrics
# ==========================================
def classification_report(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2*prec*rec / (prec + rec) if (prec + rec) else 0.0
    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1,
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}

# ==========================================
# 5. Main Execution
# ==========================================
if __name__ == '__main__':
    TRAIN_IMG_DIR = './BSR/BSDS500/data/images/train'
    TRAIN_GT_DIR = './BSR/BSDS500/data/groundTruth/train'
    TEST_IMG_DIR = './BSR/BSDS500/data/images/test'
    TEST_GT_DIR = './BSR/BSDS500/data/groundTruth/test'
    
    X_train_full, y_train_full = load_bsds500_data(TRAIN_IMG_DIR, TRAIN_GT_DIR, max_images=5)
    
    if len(X_train_full) > 0:
        X_train, y_train = balanced_sampling(X_train_full, y_train_full, samples_per_class=250)
        X_test_full, y_test_full = load_bsds500_data(TEST_IMG_DIR, TEST_GT_DIR, max_images=2)
        X_test, y_test = balanced_sampling(X_test_full, y_test_full, samples_per_class=250)
        
        print("Training Gaussian Naive Bayes Model...")
        model = GaussianNaiveBayesScratch(var_smoothing=1e-4)
        model.fit(X_train, y_train)
        
        print("Evaluating Model...")
        y_pred = model.predict(X_test)
        metrics = classification_report(y_test, y_pred)
        print("Test Metrics:", metrics)
    else:
        print("Dataset not loaded. Please ensure dataset paths are correct to run the full pipeline.")
